# PAYMENTS CLEANING

In [1239]:
import pandas as pd
payments_starter = pd.read_csv("../raw_data/sanlam_raw_payments.csv")

payments_starter.head()

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14",Debit Order,R,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed


In [1240]:
# Check to see if the transaction date has any NA values 
payments_starter['transaction_date'].isna().sum()


np.int64(162)

In [1241]:
# Step 1: Create a parser for the known formats
def parse_transaction_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    try:
        # DD/MM/YYYY
        if '/' in date_str and len(date_str.split('/')[0]) == 2:
            return pd.to_datetime(date_str, format='%d/%m/%Y')

        # MM-DD-YYYY
        elif '-' in date_str and len(date_str.split('-')[0]) == 2:
            return pd.to_datetime(date_str, format='%m-%d-%Y')

        # YYYY-MM-DD
        elif len(date_str.split('-')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y-%m-%d')

        # YYYY/MM/DD
        elif len(date_str.split('/')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y/%m/%d')

        return pd.NaT

    except:
        return pd.NaT

In [1242]:
# Step 2: Parse the dates
payments_starter['parse_transaction_date'] = (
    payments_starter['transaction_date']
    .apply(parse_transaction_date)
)

In [1243]:
payments_starter.head()

,source_row_id,transaction_ref,policy_id,client_id,transaction_date,amount,transaction_type,currency,payment_method,payment_status,source_system,parse_transaction_date
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,2022-11-15,"1,826.09",Commission,R,NaN,Reversed,LegacyLedger,2022-11-15
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,2022-10-23,"R -7,370.85",Refund,ZAR,EFT,success,PaymentsHub,2022-10-23
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,10/03/2025,"R 4,318.19",EFT,ZAR,Card,Paid,LegacyLedger,2025-03-10
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,2025/03/21,"R 4,114.14",Debit Order,R,EFT,Successful,PaymentsHub,2025-03-21
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,2025/04/23,"ZAR 21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,2025-04-23


In [1244]:
# Step 3: Find malformed dates
bad_dates = payments_starter[
    payments_starter['transaction_date'].notna() &
    payments_starter['parse_transaction_date'].isna()
]

In [1245]:
# Step 4: Count malformed dates
bad_dates.shape[0]

454

In [1249]:
#Step 5
payments_starter['transaction_date_std'] = (
    payments_starter['parse_transaction_date']
    .dt.strftime('%d-%m-%Y')
)

In [1250]:
payments_starter.columns.tolist()

['source_row_id',
 'transaction_ref',
 'policy_id',
 'client_id',
 'transaction_date',
 'amount',
 'transaction_type',
 'currency',
 'payment_method',
 'payment_status',
 'source_system',
 'parse_transaction_date',
 'transaction_date_std']

In [ ]:
payments_starter.head(30)

In [1252]:
#Rename the transaction_date_std to transaction_date_new
payments_starter.rename(columns={'transaction_date_std': 'transaction_date_new'}, inplace=True)

In [1253]:
# Remove the parse_transaction_date coloumn 
payments_starter.drop(columns=['parse_transaction_date'], inplace=True)

In [1254]:
# Remove the transaction_date coloumn (original)
payments_starter.drop(columns=['transaction_date'], inplace=True)

In [1255]:
cols = payments_starter.columns.tolist()

# Remove the column from its current position
cols.remove('transaction_date_new')

# Insert it into the 4th position
cols.insert(3, 'transaction_date_new')

# Reorder the dataframe
payments = payments_starter[cols]

In [1256]:
payments_starter.dtypes

source_row_id           str
transaction_ref         str
policy_id               str
client_id               str
amount                  str
transaction_type        str
currency                str
payment_method          str
payment_status          str
source_system           str
transaction_date_new    str
dtype: object

In [1257]:
# Convert string to datetime 
payments_starter['transaction_date_new'] = pd.to_datetime(
    payments_starter['transaction_date_new'],
    format='%d-%m-%Y',
    errors='coerce'
)

In [1258]:
## Check the malformed dates (QUARTINIE ALL 616 ROWS)
payments_starter['transaction_date_new'].isna().sum()

np.int64(616)

In [1259]:
# Check the currency distinct values 
payments_starter['currency'].unique().tolist()

['R', 'ZAR', 'USD']

In [1260]:
payments_starter['currency'] = payments_starter['currency'].replace('R', 'ZAR')

In [1261]:
# Check the distinct values for payment method. 
payments_starter['payment_method'].unique().tolist()

[nan, 'EFT', 'Card', 'Debit Order', 'Cash']

In [1262]:
payments_starter['payment_method'].isna().sum()

np.int64(12020)

In [1263]:
# Set 1202payment_method0 payment method to "Unknown"
payments_starter['payment_method'] = payments_starter['payment_method'].fillna('Unknown')

In [1264]:
payments_starter['payment_method'].head(10)

0        Unknown
1            EFT
2           Card
3            EFT
4    Debit Order
5        Unknown
6           Cash
7           Cash
8    Debit Order
9           Card
Name: payment_method, dtype: str

In [1265]:
payments_starter['payment_method'].unique().tolist()

['Unknown', 'EFT', 'Card', 'Debit Order', 'Cash']

In [1266]:
#Remove trailling spaces and whitespaces. 
payments_starter['payment_method'] = payments_starter['payment_method'].str.strip()

In [ ]:
payments_starter.head(10)

In [ ]:
# Check if the coloumns contain R or ZAR 
payments_starter[
    payments_starter['amount'].astype(str).str.contains('R|ZAR', na=False)
]

In [1269]:
#Replace the R and ZAR from amounts 
payments_starter['amount'] = payments_starter['amount'].str.replace('ZAR', '', regex=False)
payments_starter['amount'] = payments_starter['amount'].str.replace('R', '', regex=False)
payments_starter['amount'] = payments_starter['amount'].str.strip()

In [1270]:
#Check the distinct values in transcation type
payments_starter['transaction_type'].unique().tolist()

['Commission',
 'Refund',
 'EFT',
 'Debit Order',
 'Premium',
 'Reversal',
 'Commission Reversal']

In [1271]:
#Display the value counts for each transaction type
payments_starter['payment_status'].value_counts(dropna=False)

payment_status
Successful    10117
success       10039
Reversed      10020
Failed         9991
Paid           9963
Pending        9870
Name: count, dtype: int64

In [1272]:
# Merge success into successful. 
payments_starter['payment_status'] = payments_starter['payment_status'].replace({
    'success': 'Successful'
})

In [1273]:
# Merge paid into Successful
payments_starter['payment_status'] = payments_starter['payment_status'].replace({
    'Paid': 'Successful'
})

In [1274]:
payments_starter['transaction_type'] = payments_starter['transaction_type'].str.strip()

In [1275]:
payments_starter.columns

Index(['source_row_id', 'transaction_ref', 'policy_id', 'client_id', 'amount',
       'transaction_type', 'currency', 'payment_method', 'payment_status',
       'source_system', 'transaction_date_new'],
      dtype='str')

In [1276]:
#Create a status crosswalk. 
status_crosswalk = {
    'success': 'Successful',
    'Successful': 'Successful',
    'Paid': 'Paid',
    'Reversed': 'Reversed',
    'Failed': 'Failed',
    'Pending': 'Pending'
}

In [1277]:
payments_starter['payment_status'] = (
    payments_starter['payment_status']
    .map(status_crosswalk)
)

In [1278]:
crosswalk = pd.DataFrame({
    'original_value': ['success', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending'],
    'standardized_value': ['Successful', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending']
})

print(crosswalk)

  original_value standardized_value
0        success         Successful
1     Successful         Successful
2           Paid               Paid
3       Reversed           Reversed
4         Failed             Failed
5        Pending            Pending


In [1279]:
#Create a backup if anything goes wrong 
payments_backup = payments.copy()

In [1181]:
# Created a third backup 
payments_v3 = payments_backup.copy() 

In [ ]:
payments_starter.head(10)

In [1281]:
#Counts if the transcation ref appears more than once 
payments_starter['transaction_ref'].value_counts()[payments_starter['transaction_ref'].value_counts() > 1]

transaction_ref
TX-S-0000003579    4
TX-S-0000000069    3
TX-S-0000000072    3
TX-S-0000000347    3
TX-S-0000000441    3
                  ..
TX-S-0000051281    2
TX-S-0000051967    2
TX-S-0000052514    2
TX-S-0000053824    2
TX-S-0000056554    2
Name: count, Length: 696, dtype: int64

In [1282]:
payments.loc[
    payments_starter['policy_id'].astype(str).str.contains('MISSING', na=False)
]

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
16,SPAY-0000000017,TX-S-0000000017,MISSING-859967,06-09-2022,CL00001008,"R 14,281.58",EFT,R,EFT,Pending,LegacyLedger
19,SPAY-0000000020,TX-S-0000000020,MISSING-584620,22-03-2022,CL00006573,"R 11,316.51",EFT,ZAR,Debit Order,success,BankFeed
37,SPAY-0000000038,TX-S-0000000038,MISSING-745650,18-09-2025,MISSING-891886,1597.36,Premium,USD,EFT,success,PaymentsHub
179,SPAY-0000000180,TX-S-0000000180,MISSING-527227,06-04-2025,CL00007106,"ZAR 9,184.44",Commission,R,Debit Order,Reversed,LegacyLedger
236,SPAY-0000000237,TX-S-0000000237,MISSING-845014,17-10-2020,CL00001024,"10,844.02",EFT,ZAR,NaN,Pending,PaymentsHub
...,...,...,...,...,...,...,...,...,...,...,...
59751,SPAY-0000059752,TX-S-0000059752,MISSING-269132,27-01-2020,CL00009506,"(-17,539.46)",Reversal,ZAR,Card,success,PaymentsHub
59770,SPAY-0000059771,TX-S-0000059771,MISSING-220092,05-08-2022,CL00003796,"R 11,561.27",Commission,USD,Debit Order,Reversed,LegacyLedger
59849,SPAY-0000059850,TX-S-0000059850,MISSING-402395,27-08-2022,CL00011178,"16,576.43",Commission,ZAR,Debit Order,Failed,LegacyLedger
59925,SPAY-0000059926,TX-S-0000059926,MISSING-126451,01-01-2024,CL00001410,"R 1,565.66",Premium,ZAR,NaN,Successful,BankFeed


In [1185]:
#payments['policy_id_flag'] = payments['policy_id'].isin(['MISSING', 'MISSING-'])

In [1283]:
payments_starter['A04_flag'] = 'False'
 
payments_starter.loc[
    payments_starter['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'
 

In [1284]:
#Remove trailling spaces and whitespaces from amount
payments_starter['amount'] = payments_starter['amount'].str.strip()

In [1285]:
# Convert object to numeric for amount
payments_starter['amount'] = pd.to_numeric(payments_starter['amount'], errors='coerce') 

In [1286]:
payments_starter.head()

,source_row_id,transaction_ref,policy_id,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,transaction_date_new,A04_flag
0,SPAY-0000000001,TX-S-0000000001,POL000003392,CL00007794,NaN,Commission,ZAR,Unknown,Reversed,LegacyLedger,2022-11-15,False
1,SPAY-0000000002,TX-S-0000000002,POL000000796,CL00008081,NaN,Refund,ZAR,EFT,Successful,PaymentsHub,2022-10-23,False
2,SPAY-0000000003,TX-S-0000000003,POL000003511,CL00007728,NaN,EFT,ZAR,Card,Successful,LegacyLedger,2025-03-10,False
3,SPAY-0000000004,TX-S-0000000004,POL000012172,CL00001221,NaN,Debit Order,ZAR,EFT,Successful,PaymentsHub,2025-03-21,False
4,SPAY-0000000005,TX-S-0000000005,POL000014430,CL00008773,NaN,EFT,ZAR,Debit Order,Reversed,BankFeed,2025-04-23,False


In [ ]:
payments_starter[payments_starter['amount'].isna()]

# PAYMENTS_BACKUP 

In [1191]:
payments_backup.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed


In [1192]:
payments_backup['A04_flag'] = 'False'
 
payments_backup.loc[
    payments_backup['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'

In [1193]:
payments_backup.dtypes

source_row_id                      str
transaction_ref                    str
policy_id                          str
transaction_date_new    datetime64[us]
client_id                          str
amount                             str
transaction_type                   str
currency                           str
payment_method                     str
payment_status                     str
source_system                      str
A04_flag                           str
dtype: object

In [1194]:
payments_backup['transaction_date_new'].isnull()

0        False
1        False
2        False
3        False
4        False
         ...  
59995    False
59996    False
59997    False
59998    False
59999    False
Name: transaction_date_new, Length: 60000, dtype: bool

In [1195]:
# Create quarantine dataset
quarantine_date = payments_backup[payments_backup['transaction_date_new'].isna()]
 
# Remaining valid records
payments_cleaned = payments_backup[payments_backup['transaction_date_new'].notna()]

In [1196]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_date))
print('Total Records:', len(quarantine_date)+ len(payments_cleaned))
 

Original records: 59384
Quarantined records: 616
Total Records: 60000


In [1197]:
#Creating a new table for the quarantined fsp number
quarantine_date.to_csv("../quarantine/quarantine_date.csv", index=False)

In [1198]:
#Check if all the quaratined date values have moved over to the quarantine file. 
quarantine_date

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
60,SPAY-0000000061,TX-S-0000000061,POL000002607,NaT,CL00007481,"2,671.52",Debit Order,ZAR,Unknown,Successful,PaymentsHub,False
104,SPAY-0000000105,TX-S-0000000105,POL000010865,NaT,CL00002170,6054.51,Debit Order,ZAR,Card,Pending,LegacyLedger,False
112,SPAY-0000000113,TX-S-0000000113,POL000013472,NaT,CL00002934,497.45,Debit Order,ZAR,Debit Order,Successful,PaymentsHub,False
146,SPAY-0000000147,TX-S-0000000147,POL000019919,NaT,CL00008635,"18,144.84",Premium,ZAR,EFT,Successful,LegacyLedger,False
181,SPAY-0000000182,TX-S-0000000182,POL000003624,NaT,CL00000134,255.14,Premium,ZAR,Debit Order,Successful,PaymentsHub,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59297,SPAY-0000059298,TX-S-0000059298,POL000024846,NaT,CL00002672,"-9.758,08",Commission Reversal,USD,EFT,Successful,LegacyLedger,False
59401,SPAY-0000059402,TX-S-0000059402,POL000017589,NaT,CL00001185,"7,939.18",Premium,ZAR,EFT,Pending,BankFeed,False
59520,SPAY-0000059521,TX-S-0000059521,POL000025018,NaT,CL00003269,"11,979.93",Commission,ZAR,EFT,Reversed,BankFeed,False
59687,SPAY-0000059688,TX-S-0000059688,POL000017576,NaT,CL00008224,"10,545.55",Commission,ZAR,Debit Order,Reversed,PaymentsHub,False


In [1199]:
payments_cleaned['amount'].unique()

<ArrowStringArray>
[    '1,826.09',    '-7,370.85',     '4,318.19',     '4,114.14',
    '21,151.57',            nan, '(-24,653.54)',      '5285.68',
    '20,646.63',   '-21,763.22',
 ...
     '2,172.71',    '22,939.73',      '9983.90',    '19.757,32',
     '19046.91',    '13.273,09',     '9,196.56',     '6.299,68',
     '4,285.22',     '23240.21']
Length: 58441, dtype: str

In [1200]:
payments_cleaned['amount'].isnull().sum()

np.int64(359)

In [1201]:
payments_cleaned.head(10)

,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag
0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger,False
1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub,False
2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger,False
3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub,False
4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,False
5,SPAY-0000000006,TX-S-0000000006,POL000008647,2025-03-09,CL00000156,NaN,Premium,USD,Unknown,Failed,BankFeed,False
6,SPAY-0000000007,TX-S-0000000007,POL000000436,2024-06-23,CL00003805,"(-24,653.54)",Reversal,ZAR,Cash,Successful,PaymentsHub,False
7,SPAY-0000000008,TX-S-0000000008,POL000024244,2022-05-31,CL00000252,5285.68,Premium,ZAR,Cash,Successful,PaymentsHub,False
8,SPAY-0000000009,TX-S-0000000009,POL000012945,2021-03-31,CL00007328,"20,646.63",Premium,ZAR,Debit Order,Reversed,LegacyLedger,False
9,SPAY-0000000010,TX-S-0000000010,POL000010985,2025-10-10,CL00004931,"-21,763.22",Commission Reversal,ZAR,Card,Successful,BankFeed,False


In [1202]:
#Create a flag where A16_R1 is displayed. 
payments_cleaned['A16_R1'] = payments_cleaned['amount'].isna()

In [1203]:
# Create quarantine dataset
quarantine_A16_R1 = payments_cleaned[payments_cleaned['A16_R1']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R1']].copy()

In [1204]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R1))
print('Total Records:', len(quarantine_A16_R1)+ len(payments_cleaned))

Original records: 59025
Quarantined records: 359
Total Records: 59384


In [1205]:
#Creating a new table for the quarantined A16_R1 values 
quarantine_A16_R1.to_csv("../quarantine/quarantine_A16_R1.csv", index=False)

In [1206]:
#Create an A16_R3 flag where amount show "not recorded"
payments_cleaned['A16_R3'] = (
    payments_cleaned['amount']
    .str.strip()
    .str.lower()
    .eq('not recorded')
)

In [1207]:
# Create quarantine dataset
quarantine_A16_R3 = payments_cleaned[payments_cleaned['A16_R3']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R3']].copy()

In [1208]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R3))
print('Total Records:', len(quarantine_A16_R3)+ len(payments_cleaned))

Original records: 58847
Quarantined records: 178
Total Records: 59025


In [1209]:
#Creating a new table for the quarantined A16_R3 values 
quarantine_A16_R3.to_csv("../quarantine/quarantine_A16_R3.csv", index=False)

In [1210]:
payments_cleaned = payments_cleaned.to_csv("../clean_data/payments_cleaned.csv")

# CLAIMS CLEANING

In [1211]:
import pandas as pd
claims = pd.read_csv("../raw_data/sanlam_raw_claims.csv")

claims.head()

,source_row_id,claim_id,policy_id,client_id,claim_type,claim_date,reported_date,claim_amount,paid_amount,claim_status,source_system
0,SCL-000000001,CLM000000001,POL000007116,CL00009170,Disability,2021-03-22,04-29-2021,"544.958,77","ZAR 536,551.56",Paid,MergedClaims
1,SCL-000000002,CLM000000002,POL000008707,CL00003345,Disability,2025/06/01,2025/06/15,"ZAR 633,239.14","394.744,96",Paid,ClaimsOne
2,SCL-000000003,CLM000000003,POL000015185,CL00003204,Death,04-16-2021,2021/04/17,"ZAR 855,027.26","689.620,68",paid,ClaimsLegacy
3,SCL-000000004,CLM000000004,POL000016151,CL00000904,Critical Illness,2025/10/18,11-08-2025,"162,147.98",90081.68,Assessed,ClaimsOne
4,SCL-000000005,CLM000000005,POL000021628,CL00001512,disability,2020/09/06,2020-10-17,"666,398.08",587506.00,Open,ClaimsLegacy


In [1212]:
#Cleaned claims_cleaned dataset. 
claims_cleaned = pd.read_csv('../clean_data/claims_clean.csv')

claims_cleaned.head()

,source_row_id,claim_id,policy_id,client_id,claim_type,claim_date,reported_date,claim_amount,paid_amount,claim_status,source_system
0,SCL-000000001,CLM000000001,POL000007116,CL00009170,Disability,2021-03-22,2021-04-29,544958.77,536551.56,Paid,MergedClaims
1,SCL-000000002,CLM000000002,POL000008707,CL00003345,Disability,2025-06-01,2025-06-15,633239.14,394744.96,Paid,ClaimsOne
2,SCL-000000003,CLM000000003,POL000015185,CL00003204,Death,2021-04-16,2021-04-17,855027.26,689620.68,Paid,ClaimsLegacy
3,SCL-000000004,CLM000000004,POL000016151,CL00000904,Critical Illness,2025-10-18,2025-11-08,162147.98,90081.68,Assessed,ClaimsOne
4,SCL-000000005,CLM000000005,POL000021628,CL00001512,Disability,2020-09-06,2020-10-17,666398.08,587506.00,Open,ClaimsLegacy


In [1213]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   source_row_id  12000 non-null  str  
 1   claim_id       12000 non-null  str  
 2   policy_id      12000 non-null  str  
 3   client_id      11724 non-null  str  
 4   claim_type     12000 non-null  str  
 5   claim_date     11964 non-null  str  
 6   reported_date  11975 non-null  str  
 7   claim_amount   11961 non-null  str  
 8   paid_amount    11948 non-null  str  
 9   claim_status   12000 non-null  str  
 10  source_system  12000 non-null  str  
dtypes: str(11)
memory usage: 2.3 MB


In [1214]:
claims_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 11058 entries, 0 to 11057
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   source_row_id  11058 non-null  str    
 1   claim_id       11058 non-null  str    
 2   policy_id      11058 non-null  str    
 3   client_id      10797 non-null  str    
 4   claim_type     11058 non-null  str    
 5   claim_date     11058 non-null  str    
 6   reported_date  10971 non-null  str    
 7   claim_amount   11058 non-null  float64
 8   paid_amount    11058 non-null  float64
 9   claim_status   11058 non-null  str    
 10  source_system  11058 non-null  str    
dtypes: float64(2), str(9)
memory usage: 1.9 MB


# Checkout

In [1215]:
claims_cleaned.head()

,source_row_id,claim_id,policy_id,client_id,claim_type,claim_date,reported_date,claim_amount,paid_amount,claim_status,source_system
0,SCL-000000001,CLM000000001,POL000007116,CL00009170,Disability,2021-03-22,2021-04-29,544958.77,536551.56,Paid,MergedClaims
1,SCL-000000002,CLM000000002,POL000008707,CL00003345,Disability,2025-06-01,2025-06-15,633239.14,394744.96,Paid,ClaimsOne
2,SCL-000000003,CLM000000003,POL000015185,CL00003204,Death,2021-04-16,2021-04-17,855027.26,689620.68,Paid,ClaimsLegacy
3,SCL-000000004,CLM000000004,POL000016151,CL00000904,Critical Illness,2025-10-18,2025-11-08,162147.98,90081.68,Assessed,ClaimsOne
4,SCL-000000005,CLM000000005,POL000021628,CL00001512,Disability,2020-09-06,2020-10-17,666398.08,587506.00,Open,ClaimsLegacy


In [1302]:
payments_starter.to_csv("../clean_data/payments_data_cleaned.csv", index=False)


In [1288]:
payments_starter

,Unnamed: 0,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag,A16_R1,A16_R3
0,0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,"1,826.09",Commission,ZAR,Unknown,Reversed,LegacyLedger,False,False,False
1,1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,"-7,370.85",Refund,ZAR,EFT,Successful,PaymentsHub,False,False,False
2,2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,"4,318.19",EFT,ZAR,Card,Successful,LegacyLedger,False,False,False
3,3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,"4,114.14",Debit Order,ZAR,EFT,Successful,PaymentsHub,False,False,False
4,4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,"21,151.57",EFT,ZAR,Debit Order,Reversed,BankFeed,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58842,59995,SPAY-0000059996,TX-S-0000059996,POL000014233,2024-10-07,CL00008192,"13.273,09",EFT,USD,Debit Order,Successful,PaymentsHub,False,False,False
58843,59996,SPAY-0000059997,TX-S-0000059997,POL000023639,2024-12-05,CL00007282,"9,196.56",Commission,ZAR,Cash,Failed,PaymentsHub,False,False,False
58844,59997,SPAY-0000059998,TX-S-0000059998,POL000022106,2020-11-17,CL00003267,"6.299,68",Debit Order,ZAR,Cash,Pending,PaymentsHub,False,False,False
58845,59998,SPAY-0000059999,TX-S-0000059999,POL000017518,2022-11-11,CL00007067,"4,285.22",Premium,USD,Card,Pending,PaymentsHub,False,False,False


In [1289]:
payments_starter[
    payments_starter['amount']
    .astype(str)
    .str.contains('[A-Za-z]', na=False)
]['amount'].value_counts(dropna=False)
 

Series([], Name: count, dtype: int64)

In [1299]:
payments_starter['amount'] = (
    payments_starter['amount']
    .replace({
        'not recorded': pd.NA,
        'R --': pd.NA , 
        '--' :pd.NA
    })
)

In [1291]:
payments_starter['amount'] = (
    payments_starter['amount']
    .str.strip()
    .str.replace('R', '', regex=False)
    .str.replace('ZAR', '', regex=False)
    .str.replace('zar', '', regex=False)
    .str.replace('ZA', '', regex=False)
    .str.strip()
)

In [1296]:
payments_starter[
    payments_starter['amount'].str.contains(r'\(', na=False)
]['amount'].head()

Series([], Name: amount, dtype: str)

In [1293]:
mask = payments_starter['amount'].str.contains(
    r'\.\d{3},\d{2}', na=False
)
 
payments_starter.loc[mask, 'amount'] = (
    payments_starter.loc[mask, 'amount']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
 
payments_starter['amount'] = (
    payments_starter['amount']
    .str.replace(',', '', regex=False)
)

In [1294]:
payments_starter['amount'] = (
    payments_starter['amount']
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
)

In [1300]:
payments_starter['amount'].isin([
    'N/A',
    'NULL',
    'Not Recorded',
    'R --',
    '--'
]).sum()

np.int64(0)

In [1297]:
payments_starter

,Unnamed: 0,source_row_id,transaction_ref,policy_id,transaction_date_new,client_id,amount,transaction_type,currency,payment_method,payment_status,source_system,A04_flag,A16_R1,A16_R3
0,0,SPAY-0000000001,TX-S-0000000001,POL000003392,2022-11-15,CL00007794,1826.09,Commission,ZAR,Unknown,Reversed,LegacyLedger,False,False,False
1,1,SPAY-0000000002,TX-S-0000000002,POL000000796,2022-10-23,CL00008081,-7370.85,Refund,ZAR,EFT,Successful,PaymentsHub,False,False,False
2,2,SPAY-0000000003,TX-S-0000000003,POL000003511,2025-03-10,CL00007728,4318.19,EFT,ZAR,Card,Successful,LegacyLedger,False,False,False
3,3,SPAY-0000000004,TX-S-0000000004,POL000012172,2025-03-21,CL00001221,4114.14,Debit Order,ZAR,EFT,Successful,PaymentsHub,False,False,False
4,4,SPAY-0000000005,TX-S-0000000005,POL000014430,2025-04-23,CL00008773,21151.57,EFT,ZAR,Debit Order,Reversed,BankFeed,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58842,59995,SPAY-0000059996,TX-S-0000059996,POL000014233,2024-10-07,CL00008192,13273.09,EFT,USD,Debit Order,Successful,PaymentsHub,False,False,False
58843,59996,SPAY-0000059997,TX-S-0000059997,POL000023639,2024-12-05,CL00007282,9196.56,Commission,ZAR,Cash,Failed,PaymentsHub,False,False,False
58844,59997,SPAY-0000059998,TX-S-0000059998,POL000022106,2020-11-17,CL00003267,6299.68,Debit Order,ZAR,Cash,Pending,PaymentsHub,False,False,False
58845,59998,SPAY-0000059999,TX-S-0000059999,POL000017518,2022-11-11,CL00007067,4285.22,Premium,USD,Card,Pending,PaymentsHub,False,False,False
